# 04 - Cuenta, operaciones y ordenes

Lectura de la cuenta y estimacion de costos. **No se envia ninguna orden**: la
creacion y la cancelacion quedan comentadas para revisar antes de ejecutar.

## Que se ve en este notebook

- Saldos por moneda y plazo.
- Posiciones de la cartera.
- Movimientos y operaciones de trading.
- Historial de patrimonio.
- Estimacion de una orden, que no tiene efectos.
- Los metodos de escritura y su validacion local.


## 1. Credenciales

In [ ]:
import os

# completar: exportar las credenciales en el entorno antes de ejecutar.
USERNAME = os.environ.get("IOL_USERNAME", "")  # completar
PASSWORD = os.environ.get("IOL_PASSWORD", "")  # completar

if not USERNAME or not PASSWORD:
    raise SystemExit("Faltan IOL_USERNAME e IOL_PASSWORD en el entorno")


## 2. Imports y rango de fechas

In [ ]:
from datetime import datetime, timedelta

from pyIol2 import Countries, IOLAPIError, IOLClient, Markets, SettlementTerms

hasta = datetime.now()
START = (hasta - timedelta(days=30)).strftime("%Y-%m-%d")
END = hasta.strftime("%Y-%m-%d")
SYMBOL = "GGAL"


## 3. Saldos

`AccountBalances` separa pesos (`ars`) y dolares (`usd`), con totales, disponibles y comprometidos por plazo.

In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    saldos = client.get_account_balances(Countries.ARGENTINA)
    print("ARS total T0:", saldos.ars.totals.t0)
    print("ARS disponible T0:", saldos.ars.available.t0)
    print("ARS comprometido T0:", saldos.ars.committed.t0)
    print("ARS para retirar:", saldos.ars.available_for_withdrawal)
    print("USD disponible T0:", saldos.usd.available.t0)


## 4. Posiciones

`get_positions(country, valuation_currency)` devuelve `Portfolio`. La moneda de valuacion puede ser `ARS`, `USD`, `MEP` o `CCL`.

In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    cartera = client.get_positions(Countries.ARGENTINA, valuation_currency="ARS")
    print(cartera.account, cartera.currency, cartera.positions_count, "posiciones")
    print("valor de mercado:", cartera.total_market_value)
    for posicion in cartera.positions:
        print(
            posicion.instrument.symbol,
            posicion.quantity,
            posicion.market_value.current_price,
            posicion.cost_basis.average_entry_price,
            posicion.profit_loss.unrealized_pl_percent,
            posicion.portfolio_weight_percent,
        )


## 5. Movimientos y operaciones

Los movimientos son de la cuenta (depositos, retiros, rentas). Las operaciones son de trading y traen lo pedido y lo ejecutado.

In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    movimientos = client.get_account_activities(
        START,
        END,
        state="all",
        country="all",
        currency="all",
    )
    print("movimientos:", len(movimientos))
    for movimiento in movimientos[:10]:
        print(movimiento.trade_date, movimiento.activity_type, movimiento.net_amount)


In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    operaciones = client.get_trading_operations(START, END, state="all")
    print("operaciones:", len(operaciones))
    for operacion in operaciones[:10]:
        print(
            operacion.order_date,
            operacion.type,
            operacion.symbol,
            operacion.status,
            operacion.order.qty,
            operacion.executed.qty,
        )


## 6. Historial de patrimonio

Si `status` es `partial`, revisar `missing_dates`: la API no informo esos dias.

In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    historial = client.get_portfolio_history(START, END, Countries.ARGENTINA)
    print("status:", historial.status, "| dias faltantes:", historial.missing_dates)
    for punto in historial.history[-5:]:
        print(punto.date, punto.equity.ars, punto.equity.usd_ccl)


## 7. Estimacion de una orden

`estimate_order()` calcula importe, comision y cargos regulatorios. **No envia
nada.** Reglas que valida el cliente antes de llamar a la API:

- `quantity` o `notional`, pero no los dos.
- `limit_price` obligatorio cuando `order_type="limit"`.


In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    estimacion = client.estimate_order(
        SYMBOL,
        "buy",
        quantity=1,
        market_id=Markets.BCBA,
        order_type="market",
        settlement_term=SettlementTerms.T0,
    )
    print("importe:", estimacion.amount.value, estimacion.amount.currency)
    print("comision:", estimacion.commission.value, estimacion.commission.currency)
    print(
        "derecho de mercado:",
        estimacion.regulatory_fees.market_fee.value,
        estimacion.regulatory_fees.market_fee.currency,
    )
    print("IVA:", estimacion.regulatory_fees.vat.value)


## 8. Validacion local

Un cuerpo invalido falla con `IOLValidationError` antes de salir a la red: la request nunca se envia.

In [ ]:
from pyIol2 import IOLValidationError

try:
    with IOLClient(USERNAME, PASSWORD) as client:
        client.estimate_order(SYMBOL, "buy", order_type="limit")
except IOLValidationError as exc:
    print("rechazado antes de enviar:", exc.message)


## 9. Escritura (comentado)

`create_order()` y `cancel_order()` **mueven dinero real**. Antes de
descomentar:

- Verificar simbolo, lado, mercado, plazo, cantidad y precio.
- Revisar la estimacion del paso 7 y el estado de la cuenta.
- Pasar un `idempotency_key` propio si el sistema puede reintentar: sin clave,
  el cliente genera una nueva en cada llamada.
- No reintentar una escritura incierta: primero buscar la operacion con
  `get_trading_operations()` o `get_account_activities()`.

Los metodos de FCI y cauciones siguen la misma regla.


In [ ]:
# with IOLClient(USERNAME, PASSWORD) as client:
#     orden = client.create_order(
#         SYMBOL,
#         "buy",
#         quantity=1,
#         market_id=Markets.BCBA,
#         order_type="market",
#         settlement_term=SettlementTerms.T0,
#         idempotency_key="<uuid-v4-de-la-intencion>",
#     )
#     print("orden:", orden.order_id, orden.successful_purchase)
#     for mensaje in orden.messages:
#         print(mensaje.code, mensaje.message)
#
#     cancelacion = client.cancel_order(orden.order_id)
#     print("cancelada:", cancelacion.cancelled, cancelacion.status)

print("bloque de escritura comentado")


## 10. Errores

Todo error de la API llega como `IOLAPIError` o una subclase. Registrar siempre el `trace_id`.

In [ ]:
try:
    with IOLClient(USERNAME, PASSWORD) as client:
        client.get_positions("XX")
except IOLAPIError as exc:
    print(type(exc).__name__, exc.status_code, exc.code)
    print("trace_id:", exc.trace_id)
